# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

print(f"Rows:          {len(df):,}")
print(f"Total revenue: ${df['revenue'].sum():,.2f}")
print(f"Total units:   {df['qty'].sum():,}")

Rows:          400
Total revenue: $8,520.00
Total units:   783


I added a `revenue` column found by multiplying the quantity of items sold by the price. Then to get the total revenue, I summed the `revenue` column. This found that 783 items were sold, which made a total revenue of $8,520

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = (df.groupby('category', as_index=False)['revenue'].sum()
            .sort_values('revenue', ascending=False)
            .reset_index(drop=True))
by_category['share_percent'] = (by_category['revenue'] / df['revenue'].sum() * 100).round(1)
by_category

,category,revenue,share_percent
0,Food,4293.0,50.4
1,Merch,1771.5,20.8
2,Drink,1554.0,18.2
3,RainGear,901.5,10.6


I grouped by `category` and summed `revenue`, then sorted from highest to lowest. Then I added a `share_percent` column by dividing each category's revenue by the total revenue, and rounding to one decimal. This found that food accounted for over half of total revenue, then merch, drink, and rain gear.
These four shares sum to 100%.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
by_vendor = (df.groupby('vendor_id')['revenue']
               .agg(avg_order_revenue='mean', orders='count')
               .sort_values('avg_order_revenue', ascending=False)
               .round({'avg_order_revenue': 2}))
by_vendor

,avg_order_revenue,orders
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


First, I grouped by `vendor_id` and then computed two aggregates on `revenue`, the mean/average order revenue and the count/number of orders. Then, I sorted by the average from highest to lowest and rounded it to two decimals. V-01 is highest at $22.60, but all four vendors have about 100 orders and the averages are close, so the differences are small.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_share = df.loc[df['category'] == 'Merch', 'revenue'].sum() / df['revenue'].sum() * 100
print(f"{merch_share:.1f}%")

20.8%


I filtered for rows where `category` is `'Merch'`, added up their revenue, and then divided by total revenue to get Merch's share of revenue. Then, I multiplied by 100 and printed it rounded to one decimal, which gives 20.8%.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

n_before, rev_before = len(df), df['revenue'].sum()

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Proof that the join changed nothing
assert len(joined) == n_before
assert np.isclose(joined['revenue'].sum(), rev_before)
print(f"Rows: {n_before} -> {len(joined)} | Revenue: {rev_before:,.2f} -> {joined['revenue'].sum():,.2f}")

# Report the unmatched vendor
unmatched = joined.loc[joined['vendor_name'].isna()]
print(unmatched['vendor_id'].value_counts())
print(f"Unmatched revenue: ${unmatched['revenue'].sum():,.2f}")

# Keep and label the order
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (' + joined['vendor_id'] + ')')

Rows: 400 -> 400 | Revenue: 8,520.00 -> 8,520.00
vendor_id
V-18    108
Name: count, dtype: int64
Unmatched revenue: $2,349.00


First I saved the row count and revenue total. Then I used a left-join to merge the vendor names onto the orders with `validate='many_to_one'`. Then I used two asserts and a print line to prove that the join didn't change anything. This showed that rows stayed at 400 and revenue stayed at $8,520.

**The unmatched vendor, and what I did about it:** V-18 is in the orders, having 108 orders and $2,349 in revenue, but was not in the lookup, so it had no name. I kept those orders and labeled them "Unknown (V-18)" instead of dropping them, so the row count and revenue total stay unchanged and no important information is lost.


### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = joined.pivot_table(index='vendor_name', columns='category', values='revenue',
                           aggfunc='sum', margins=True, margins_name='Total')
pivot.style.format('{:,.2f}')

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.50,"1,054.50",400.50,175.50,"2,133.00"
Hoos Burgers,171.00,"1,338.00",373.50,241.50,"2,124.00"
Rotunda Tacos,298.50,882.00,489.00,244.50,"1,914.00"
Unknown (V-18),582.00,"1,018.50",508.50,240.00,"2,349.00"
Total,"1,554.00","4,293.00","1,771.50",901.50,"8,520.00"


I built a pivot table of `revenue` with vendors down the side and categories across the top, summing each cell. `margins=True` adds the row and column totals, and the grand total.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

I would tell vendors to prioritize selling food, because it accounted for 50.4% of all revenue, making $4,293. Furthermore, I would especially recommend Rotunda Tacos to focus more on food. They only made 882 dollars in food, significantly lower than the other restraunts who all made above 1,000 dollars. It had fewer food orders, 37, compared to the other vendors who had between 43-55 orders. Therefore, selling more food would target the biggest category where Rotunda Tacos is falling behind and help them succeed. I would also recommend that these stores do not prioritize rain gear, as it made the least revenue, only totaling 901.50 dollars, only 10.6% of all revenue. Finally, I would also recommend that Hoos Burgers try pairing drinks with its food. It made only 171 dollars in drinks, compared with 502.50 dollars that Cav Merch North made and 582 dollars that V-18 made.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

In my opinion, the least trustworthy answer is the pivot table. I chose this because some of its cells are built from very few orders. For example, each vendor has about 100 orders, and `RainGear` is only about 10% of orders, so therefore each `RainGear` cell relies on about ten orders. And using this same logic, `Drink` cells rely on about 20 orders. This is problematic because a few big orders could have a large impact on which vendors look good in these cells.
